<a href="https://colab.research.google.com/github/PaoloniCoder/MozartAI/blob/main/m3_validazione.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# M3 · Issue 2 · Validazione del convertitore con i .mid originali di PrIMuS

Per ogni esempio confrontiamo il MIDI che produce il nostro convertitore (`tokens_to_midi`) con il `.mid` originale
che sta nella stessa cartella di PrIMuS, **in tre livelli**: prima le **altezze**, poi **quando** partono le note
(in proporzione), poi **quanto durano** (in proporzione).

Come si usa: esegui le celle in ordine. Se cambi `src/midi_convert.py` (commit e push), riesegui dalla **cella 1**.

## 1 · Preparazione (scarica il repo e importa il convertitore)

In [1]:
import sys, os, glob, json, re, collections, random, importlib
from google.colab import drive
drive.mount('/content/drive')
!pip -q install pretty_midi

UTENTE = 'martinopaoloni-ux'
BRANCH = 'main'            # il branch dove sta src/midi_convert.py (se lavori su un altro branch, scrivilo qui)

!rm -rf /content/MozartAI
!git clone -b {BRANCH} https://github.com/{UTENTE}/MozartAI.git /content/MozartAI
percorsi = glob.glob('/content/MozartAI/**/midi_convert.py', recursive=True)
assert percorsi, "midi_convert.py non trovato. File .py nel repo: " + str(glob.glob('/content/MozartAI/**/*.py', recursive=True))
sys.path.insert(0, os.path.dirname(percorsi[0]))
import midi_convert
importlib.reload(midi_convert)               # così prende la versione appena scaricata
from midi_convert import tokens_to_midi, analizza_token
print("Import ok:", percorsi[0])

Mounted at /content/drive
Cloning into '/content/MozartAI'...
remote: Enumerating objects: 120, done.
remote: Counting objects: 100% (120/120), done.
remote: Compressing objects: 100% (113/113), done.
remote: Total 120 (delta 51), reused 8 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (120/120), 7.91 MiB | 10.06 MiB/s, done.
Resolving deltas: 100% (51/51), done.
Import ok: /content/MozartAI/src/midi_convert.py


## 2 · Scelta dei 200 esempi a caso
Estraggo dall'archivio solo i file `.semantic` e `.mid` (circa 1-2 minuti la prima volta).

In [2]:
import pandas as pd
import pretty_midi

ARCHIVE = '/content/drive/MyDrive/MozartAI/data/primus.tgz'
DATI = '/content/primus_mid'
N_ESEMPI = 200      # per la prova finale puoi alzarlo (es. 2000)
SEED = 42

if not os.path.exists(DATI + '/.ok'):
    os.makedirs(DATI, exist_ok=True)
    !tar -xzf "{ARCHIVE}" -C "{DATI}" --exclude='._*' --wildcards '*.semantic' '*.mid' '*.midi'
    open(DATI + '/.ok', 'w').close()

coppie = []
for radice, _, files in os.walk(DATI):
    sem = [f for f in files if f.endswith('.semantic') and not f.startswith('._')]
    mid = [f for f in files if f.lower().endswith(('.mid', '.midi')) and not f.startswith('._')]
    if sem and mid:
        coppie.append((os.path.join(radice, sem[0]), os.path.join(radice, mid[0])))
coppie.sort()
print(len(coppie), "esempi con .semantic e .mid")
assert coppie, "Nessun .mid trovato nell'archivio: controlla con  !find /content/primus_mid | head"
campione = random.Random(SEED).sample(coppie, min(N_ESEMPI, len(coppie)))
print("Esempi nel campione:", len(campione))

tar: *.midi: Not found in archive
tar: Exiting with failure status due to previous errors
87678 esempi con .semantic e .mid
Esempi nel campione: 200


## 3 · Le funzioni di confronto
- `confronta` dà tre risposte (altezze / inizi / durate) per un esempio. Il confronto è **in proporzione**: il tempo (battiti al minuto) del file originale può essere diverso dal nostro.
- `causa` prova a spiegare perché un esempio è diverso: ripete la conversione con le opzioni "armatura applicata" e "appoggiature suonate" e guarda se così torna identico.

In [3]:
def note_da_pm(pm):
    """PrettyMIDI -> lista (altezza, inizio, fine) in secondi, ordinata nel tempo (strumenti a percussione esclusi)."""
    note = []
    for strumento in pm.instruments:
        if getattr(strumento, 'is_drum', False):
            continue
        note += [(n.pitch, n.start, n.end) for n in strumento.notes]
    note.sort(key=lambda x: (round(x[1], 3), x[0]))
    return note


def confronta(mio, orig, tol_inizio=0.01, tol_durata=0.20):
    """mio, orig: liste (altezza, inizio, fine) in secondi.

    altezze: stessa lista di altezze, nello stesso ordine.
    inizi:   gli istanti di partenza coincidono IN PROPORZIONE (si scala il tempo; tolleranza = 1% della lunghezza totale).
    durate:  le durate coincidono in proporzione (si accetta che l'originale accorci tutte le note nello stesso modo; tolleranza 20%).
    """
    ris = dict(n_mio=len(mio), n_orig=len(orig), altezze=False, inizi=False, durate=False, prima_diff=None)
    if not mio or not orig:
        return ris
    pm_, po_ = [x[0] for x in mio], [x[0] for x in orig]
    for i in range(min(len(pm_), len(po_))):
        if pm_[i] != po_[i]:
            ris['prima_diff'] = i
            break
    else:
        if len(pm_) != len(po_):
            ris['prima_diff'] = min(len(pm_), len(po_))
    if len(mio) != len(orig):
        return ris
    ris['altezze'] = pm_ == po_
    # inizi in proporzione: posizione di ogni nota come frazione della lunghezza totale (dal primo all'ultimo inizio)
    sm = [x[1] - mio[0][1] for x in mio]
    so = [x[1] - orig[0][1] for x in orig]
    tm, to = sm[-1], so[-1]
    if tm > 1e-9 and to > 1e-9:
        ris['inizi'] = all(abs(a / tm - b / to) <= tol_inizio for a, b in zip(sm, so))
    else:
        ris['inizi'] = len(mio) == 1 or (tm <= 1e-9 and to <= 1e-9)
    # durate in proporzione: rapporto orig/mio di ogni nota, confrontato con il rapporto tipico (mediana)
    dm = [x[2] - x[1] for x in mio]
    do = [x[2] - x[1] for x in orig]
    if all(d > 1e-9 for d in dm) and all(d > 1e-9 for d in do):
        r = sorted(b / a for a, b in zip(dm, do))
        tip = r[len(r) // 2]
        ris['durate'] = all(abs(b / a / tip - 1) <= tol_durata for a, b in zip(dm, do))
    return ris


VARIANTI = [
    ('armatura applicata', dict(applica_armatura=True)),
    ('appoggiature suonate', dict(gracenote='breve')),
    ('armatura + appoggiature', dict(applica_armatura=True, gracenote='breve')),
]


def converti(tokens, **opzioni):
    avvisi = []
    pm = tokens_to_midi(tokens, tempo=100, avvisi=avvisi, **opzioni)
    return note_da_pm(pm), avvisi


def causa(tokens, orig, base, avvisi):
    """Etichetta la causa di un esempio non identico."""
    # 1) una variante del convertitore lo rende identico?
    for nome, opz in VARIANTI:
        mio, _ = converti(tokens, **opz)
        r = confronta(mio, orig)
        if r['altezze'] and r['inizi'] and r['durate']:
            return nome
    # 2) indizi dai simboli presenti
    if any(a.startswith('Simbolo') for a in avvisi):
        return 'simboli ignorati'
    if base['n_mio'] != base['n_orig']:
        if any(t.startswith('gracenote') for t in tokens):
            return 'numero di note diverso (con appoggiature)'
        if 'tie' in tokens:
            return 'numero di note diverso (con legature)'
        if any(t.startswith('multirest') for t in tokens):
            return 'numero di note diverso (con multirest)'
        return 'numero di note diverso (altro)'
    if not base['altezze']:
        return 'altezze diverse (altro)'
    if not base['inizi'] or not base['durate']:
        if any(t.startswith('note-') and '.' in t.split('_', 1)[-1] for t in tokens):
            return 'durate diverse (con punti di valore)'
        if 'tie' in tokens:
            return 'durate diverse (con legature)'
        return 'durate diverse (altro)'
    return 'altro'

## 4 · Confronto sui 200 esempi
Leggo i `.mid` originali una volta sola e li tengo in memoria (servono anche per le prove con le varianti).

In [4]:
dati = []          # (id, simboli, note dell'originale oppure None se illeggibile)
for sem_path, mid_path in campione:
    tokens = open(sem_path, encoding='utf-8').read().split()
    try:
        orig = note_da_pm(pretty_midi.PrettyMIDI(mid_path))
    except Exception as e:
        orig = None
    dati.append((os.path.basename(os.path.dirname(sem_path)), tokens, orig))
print("MIDI originali illeggibili:", sum(1 for d in dati if d[2] is None))

righe = []
for nome, tokens, orig in dati:
    if orig is None:
        righe.append(dict(id=nome, n_token=len(tokens), esito='.mid illeggibile', causa='.mid illeggibile'))
        continue
    mio, avvisi = converti(tokens)
    base = confronta(mio, orig)
    identico = base['altezze'] and base['inizi'] and base['durate']
    righe.append(dict(id=nome, n_token=len(tokens), **base, identico=identico,
                      causa='identico' if identico else causa(tokens, orig, base, avvisi),
                      n_avvisi=len(avvisi)))
df = pd.DataFrame(righe)
print("Fatto:", len(df), "esempi")

MIDI originali illeggibili: 0
Fatto: 200 esempi


## 5 · Risultati: percentuali e cause

In [5]:
valide = df[df['causa'] != '.mid illeggibile']
N = len(valide)
def pct(col):
    return 100 * valide[col].astype(bool).sum() / N

print(f"Esempi confrontati: {N}")
print(f"Altezze identiche                 : {pct('altezze'):5.1f}%")
print(f"Inizi identici (in proporzione)   : {pct('inizi'):5.1f}%")
print(f"Durate identiche (in proporzione) : {pct('durate'):5.1f}%")
print(f"ESEMPI IDENTICI (tutti e tre)     : {pct('identico'):5.1f}%")
print()
print("Cause degli esempi non identici:")
print(valide['causa'].value_counts().to_string())

Esempi confrontati: 200
Altezze identiche                 :  32.0%
Inizi identici (in proporzione)   :  90.5%
Durate identiche (in proporzione) :  91.5%
ESEMPI IDENTICI (tutti e tre)     :  31.0%

Cause degli esempi non identici:
causa
altezze diverse (altro)                      119
identico                                      62
numero di note diverso (con legature)         15
durate diverse (altro)                         2
numero di note diverso (con appoggiature)      2


## 6 · Alcuni esempi per ogni causa
Per ogni causa mostro fino a 2 esempi: i simboli, le altezze del nostro MIDI e quelle dell'originale, e il punto della prima differenza.

In [6]:
def mostra_esempio(riga, tokens, orig, max_n=40):
    mio, avvisi = converti(tokens)
    i = riga['prima_diff']
    i = 0 if pd.isna(i) else int(i)           # pandas lo trasforma in numero decimale (o vuoto)
    a, b = max(0, i - 4), i + 8
    print(f"--- {riga['id']} | note mio={len(mio)} originale={len(orig)} | prima differenza alla nota n.{i}")
    print("SIMBOLI :", ' '.join(tokens[:max_n]), '...' if len(tokens) > max_n else '')
    print("MIO     :", [x[0] for x in mio][a:b])
    print("ORIGINALE:", [x[0] for x in orig][a:b])
    if avvisi:
        print("AVVISI  :", avvisi[:3])

indice = {nome: (tokens, orig) for nome, tokens, orig in dati}
for c in valide['causa'].value_counts().index:
    if c == 'identico':
        continue
    print(f"\n=========== {c} ===========")
    for _, riga in valide[valide['causa'] == c].head(2).iterrows():
        tokens, orig = indice[riga['id']]
        mostra_esempio(riga, tokens, orig)


=========== altezze diverse (altro) ===========
--- 230002354-1_2_1 | note mio=14 originale=14 | prima differenza alla nota n.7
SIMBOLI : clef-G2 keySignature-FM timeSignature-3/4 note-F4_quarter note-A4_quarter note-C5_quarter barline note-F5_quarter. note-E5_sixteenth note-D5_sixteenth gracenote-D5_sixteenth note-C5_eighth note-Bb4_sixteenth note-A4_sixteenth barline gracenote-C5_sixteenth note-Bb4_sixteenth note-A4_sixteenth note-Bb4_sixteenth note-C5_sixteenth note-Bb4_quarter 
MIO     : [77, 76, 74, 72, 70, 69, 70, 69, 70, 72, 70]
ORIGINALE: [77, 76, 74, 72, 71, 69, 71, 69, 71, 72, 71]
AVVISI  : ['2 appoggiature (gracenote) ignorate']
--- 000102515-1_1_1 | note mio=15 originale=15 | prima differenza alla nota n.0
SIMBOLI : clef-G2 keySignature-DM timeSignature-2/4 note-F#5_quarter note-E5_eighth note-A5_eighth barline note-F#5_eighth note-D5_eighth note-B4_quarter barline note-E5_eighth note-C#5_eighth note-A4_eighth note-B4_sixteenth note-C#5_sixteenth barline note-D5_eighth not

## 7 · Simboli che il convertitore ignora
Tre gruppi: quelli che **non riconosce**, quelli che **ignora per scelta** (appoggiature), quelli che **non producono suoni** per natura.

In [7]:
non_riconosciuti = collections.Counter()
per_famiglia = collections.Counter()
for nome, tokens, orig in dati:
    for t in tokens:
        per_famiglia[t.split('-')[0]] += 1
        tipo = analizza_token(t)['tipo']
        if tipo in ('sconosciuto', 'malformato'):
            non_riconosciuti[t] += 1

print("1) NON riconosciuti (saltati con avviso):")
print("   ", dict(non_riconosciuti) if non_riconosciuti else "nessuno")
print("2) Ignorati per scelta:  gracenote ->", per_famiglia.get('gracenote', 0), "occorrenze nel campione")
print("3) Non producono suoni:  clef:", per_famiglia.get('clef', 0), "| barline:", per_famiglia.get('barline', 0),
      "| timeSignature:", per_famiglia.get('timeSignature', 0), "| keySignature:", per_famiglia.get('keySignature', 0))
print("\nTutte le famiglie nel campione:", dict(per_famiglia.most_common()))

1) NON riconosciuti (saltati con avviso):
    nessuno
2) Ignorati per scelta:  gracenote -> 76 occorrenze nel campione
3) Non producono suoni:  clef: 203 | barline: 796 | timeSignature: 200 | keySignature: 152

Tutte le famiglie nel campione: {'note': 3075, 'barline': 796, 'rest': 325, 'clef': 203, 'timeSignature': 200, 'keySignature': 152, 'gracenote': 76, 'multirest': 54, 'tie': 28}


## 8 · Quale impostazione è migliore?
Ripeto il confronto cambiando le opzioni del convertitore: la tabella dice quale combinazione dà più esempi identici.

In [8]:
def percentuale(**opzioni):
    ok = tot = 0
    for nome, tokens, orig in dati:
        if orig is None:
            continue
        mio, _ = converti(tokens, **opzioni)
        r = confronta(mio, orig)
        tot += 1
        ok += bool(r['altezze'] and r['inizi'] and r['durate'])
    return 100 * ok / max(tot, 1)

prove = [('predefinito (nessuna armatura, appoggiature saltate)', {}),
         ('armatura applicata', dict(applica_armatura=True)),
         ('appoggiature suonate', dict(gracenote='breve')),
         ('armatura + appoggiature', dict(applica_armatura=True, gracenote='breve'))]
for nome, opz in prove:
    print(f"{percentuale(**opz):5.1f}%  {nome}")

 31.0%  predefinito (nessuna armatura, appoggiature saltate)
 30.5%  armatura applicata
 24.0%  appoggiature suonate
 23.5%  armatura + appoggiature


## 9 · Testo da copiare nella Pull Request e in `docs/m3-appunti.md`

In [9]:
print(f"Su {N} esempi casuali di PrIMuS (seed {SEED}), confrontando il MIDI del convertitore con il .mid originale:")
print(f"- altezze identiche: {pct('altezze'):.1f}%")
print(f"- inizi identici in proporzione: {pct('inizi'):.1f}%")
print(f"- durate identiche in proporzione: {pct('durate'):.1f}%")
print(f"- esempi identici (tutti e tre i livelli): {pct('identico'):.1f}%")
print("Cause degli esempi diversi:")
for c, n in valide['causa'].value_counts().items():
    if c != 'identico':
        print(f"  - {c}: {n}")
print("Simboli non riconosciuti:", dict(non_riconosciuti) if non_riconosciuti else "nessuno")
print("Ignorati per scelta: gracenote (", per_famiglia.get('gracenote', 0), "occorrenze )")

df.to_csv('/content/drive/MyDrive/MozartAI/m3_validazione.csv', index=False)     # salvato su Drive, non nel repo

Su 200 esempi casuali di PrIMuS (seed 42), confrontando il MIDI del convertitore con il .mid originale:
- altezze identiche: 32.0%
- inizi identici in proporzione: 90.5%
- durate identiche in proporzione: 91.5%
- esempi identici (tutti e tre i livelli): 31.0%
Cause degli esempi diversi:
  - altezze diverse (altro): 119
  - numero di note diverso (con legature): 15
  - durate diverse (altro): 2
  - numero di note diverso (con appoggiature): 2
Simboli non riconosciuti: nessuno
Ignorati per scelta: gracenote ( 76 occorrenze )
